# Svarga.ai — Google Colab Workbench

**Owner: Aditya Mohan Menon. Svarga.ai — all rights reserved.**

Run this notebook top to bottom. It gives you four things:

1. **Talk to Svarga** from Python.
2. **Feed data in** — push your PDFs, EPUBs and notes into Svarga's knowledge library so it answers from them.
3. **Export training data** — pull your own conversations out as a clean instruction dataset.
4. **Fine-tune** — train a LoRA adapter on an open-source base using that dataset (free Colab T4 works).

No API key is stored in this notebook. Nothing here is shared with anyone else.


## 0. Setup

In [ ]:
!pip -q install requests supabase==2.* tqdm

SVARGA_URL = "https://svarga.digital"

# Public (safe-to-share) backend values for Svarga.
SUPABASE_URL = "https://oitfmzvlrpyjqgiqoqtq.supabase.co"
SUPABASE_ANON_KEY = "sb_publishable_EZeOoecOTNsfgf8B_RYBdg_jhKswHIj"

print("Ready.")

## 1. Talk to Svarga

This calls the live site. Signed out you get the guest allowance; sign in (section 2) for your full plan limits.

In [ ]:
import json, requests

ACCESS_TOKEN = None  # filled in by section 2 if you sign in

def ask_svarga(prompt, mode="balanced", persona=None):
    """Send one question to Svarga and stream back the answer."""
    headers = {"Content-Type": "application/json"}
    if ACCESS_TOKEN:
        headers["Authorization"] = f"Bearer {ACCESS_TOKEN}"

    payload = {
        "messages": [{"id": "1", "role": "user", "parts": [{"type": "text", "text": prompt}]}],
        "mode": mode,
    }
    if persona:
        payload["persona"] = persona

    # No timeout: reasoning answers legitimately take a while.
    with requests.post(f"{SVARGA_URL}/api/svarga-chat", headers=headers,
                       data=json.dumps(payload), stream=True) as r:
        if r.status_code == 429:
            print("Rate limited. Wait", r.headers.get("Retry-After", "60"), "seconds.")
            return ""
        if r.status_code >= 400:
            print("Error", r.status_code, r.text[:300])
            return ""

        answer = []
        for raw in r.iter_lines(decode_unicode=True):
            if not raw or not raw.startswith("data: "):
                continue
            chunk = raw[6:]
            if chunk == "[DONE]":
                break
            try:
                event = json.loads(chunk)
            except json.JSONDecodeError:
                continue
            if event.get("type") == "text-delta" and event.get("delta"):
                print(event["delta"], end="")
                answer.append(event["delta"])
        print()
        return "".join(answer)

ask_svarga("In two lines: what is Svarga.ai?")

In [ ]:
# Baby Krishna buddy, grounded in Gita principles
ask_svarga("I am anxious about my exams. What would you tell me?", persona="krishna")

## 2. Sign in

Use the same email and password you use on svarga.digital. Colab will hide the password as you type.

In [ ]:
from getpass import getpass
from supabase import create_client

email = input("Email: ")
password = getpass("Password: ")

sb = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)
session = sb.auth.sign_in_with_password({"email": email, "password": password})

ACCESS_TOKEN = session.session.access_token
USER_ID = session.user.id
print("Signed in as", session.user.email)

## 3. Feed data into Svarga

Upload a PDF, EPUB or text file. Svarga splits it, indexes it, and from then on answers your
questions from that document with citations.

Upload the file to Colab first (folder icon on the left), then set `FILE_PATH`.

In [ ]:
import os, uuid, mimetypes

FILE_PATH = "/content/my-book.pdf"   # <-- change this
TITLE     = os.path.basename(FILE_PATH)

def kind_for(path):
    ext = path.lower().rsplit(".", 1)[-1]
    return {"pdf": "pdf", "epub": "epub", "txt": "text", "md": "text"}.get(ext, "other")

def feed_document(path, title=None):
    title = title or os.path.basename(path)
    doc_kind = kind_for(path)
    storage_path = f"{USER_ID}/{uuid.uuid4()}"

    with open(path, "rb") as fh:
        sb.storage.from_("documents").upload(
            storage_path, fh.read(),
            {"content-type": mimetypes.guess_type(path)[0] or "application/octet-stream"},
        )

    row = sb.table("documents").insert({
        "user_id": USER_ID,
        "title": title,
        "file_path": storage_path,
        "kind": doc_kind,
        "status": "pending",
    }).execute()

    doc_id = row.data[0]["id"]
    print("Uploaded:", title, "->", doc_id)
    print("Open svarga.digital -> Library and press Index to make it searchable.")
    return doc_id

# feed_document(FILE_PATH, TITLE)

In [ ]:
# Feed a whole folder at once
import glob

def feed_folder(folder):
    ids = []
    for path in sorted(glob.glob(f"{folder}/*")):
        if kind_for(path) == "other":
            continue
        ids.append(feed_document(path))
    return ids

# feed_folder("/content/library")

## 4. Export your conversations as training data

This pulls **your own** saved chats and writes them as JSONL instruction pairs —
the format every fine-tuning tool expects. Nobody else's data is touched.

In [ ]:
import json

OUT = "/content/svarga-train.jsonl"

def export_dataset(out=OUT, limit=5000):
    rows = (sb.table("messages")
              .select("conversation_id, role, content, created_at")
              .order("created_at")
              .limit(limit)
              .execute().data or [])

    pairs, pending = [], None
    for row in rows:
        if row["role"] == "user":
            pending = row["content"]
        elif row["role"] == "assistant" and pending:
            pairs.append({"instruction": pending, "output": row["content"]})
            pending = None

    with open(out, "w", encoding="utf-8") as fh:
        for pair in pairs:
            fh.write(json.dumps(pair, ensure_ascii=False) + "\n")

    print(len(pairs), "examples ->", out)
    return out

# export_dataset()

### Clean the dataset before training

Garbage in, garbage out. Drop anything short, broken, or containing personal details.

In [ ]:
import re

def clean(in_path=OUT, out_path="/content/svarga-train-clean.jsonl"):
    PII = re.compile(r"(\+?\d{10,}|[\w.\-]+@[\w\-]+\.\w+)")
    kept = 0
    with open(in_path, encoding="utf-8") as src, open(out_path, "w", encoding="utf-8") as dst:
        for line in src:
            rec = json.loads(line)
            if len(rec["instruction"]) < 8 or len(rec["output"]) < 40:
                continue
            if PII.search(rec["instruction"]) or PII.search(rec["output"]):
                continue
            dst.write(json.dumps(rec, ensure_ascii=False) + "\n")
            kept += 1
    print(kept, "clean examples ->", out_path)
    return out_path

# clean()

## 5. Fine-tune an open-source model (LoRA)

**Runtime → Change runtime type → T4 GPU** before running this.

This trains a small adapter on top of an open-source base so it answers in Svarga's voice.
It does not touch the live site — the adapter is yours to test, keep, or deploy later.

Check the base model's licence before any commercial use.

In [ ]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE = "unsloth/llama-3.1-8b-instruct-bnb-4bit"   # swap for a Sarvam/Indic base if you prefer
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ, load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
from datasets import load_dataset

SVARGA_VOICE = (
    "You are Svarga, an Indian AI assistant created by Aditya Mohan Menon. "
    "Answer warmly, directly and accurately. Lead with the answer. "
    "Use Indian conventions: rupees, lakh/crore, IST, DD/MM/YYYY. "
    "Never invent a scripture verse or a citation."
)

def to_chat(rec):
    text = tokenizer.apply_chat_template([
        {"role": "system",    "content": SVARGA_VOICE},
        {"role": "user",      "content": rec["instruction"]},
        {"role": "assistant", "content": rec["output"]},
    ], tokenize=False)
    return {"text": text}

ds = load_dataset("json", data_files="/content/svarga-train-clean.jsonl", split="train")
ds = ds.map(to_chat)
print(ds)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field="text", max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=2, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="/content/svarga-lora",
    ),
)

trainer.train()

In [ ]:
# Try the tuned model
FastLanguageModel.for_inference(model)

prompt = tokenizer.apply_chat_template([
    {"role": "system", "content": SVARGA_VOICE},
    {"role": "user",   "content": "Explain SIP investing to a first-time earner in Kochi."},
], tokenize=False, add_generation_prompt=True)

out = model.generate(**tokenizer(prompt, return_tensors="pt").to("cuda"), max_new_tokens=400)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# Save the adapter (a few hundred MB, not the whole model)
model.save_pretrained("/content/svarga-lora-adapter")
tokenizer.save_pretrained("/content/svarga-lora-adapter")

!zip -qr /content/svarga-lora-adapter.zip /content/svarga-lora-adapter
print("Download /content/svarga-lora-adapter.zip and keep it safe. It is your property.")

## 6. Before you ship a tuned model

- Compare it against the live Svarga on the same 20 questions. Keep the winner.
- A LoRA adapter needs a GPU endpoint to serve (Together, Fireworks, Modal, or your own vLLM box).
  The live site's edge runtime cannot host weights.
- Route only the questions it is genuinely better at; leave the rest on the current model.

---

© Svarga.ai — owned exclusively by Aditya Mohan Menon. All rights reserved.
No part of this notebook, its prompts, datasets or resulting weights may be redistributed
or relicensed without his written permission.